## Create DataFrame from last epoch

In [18]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ── Pick the base folder that actually contains your JSONs ──
candidate_bases = [
    Path("/mnt/sharedstorage/sabdelazim/Desktop/kaitlyn_catalyst/notebooks"),
    Path("/mnt/sharedstorage/sabdelazim/Desktop/kaitlyn_catalyst/speciesnet"),
    Path("/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/speciesnet"),
    Path.cwd(),
]
for b in candidate_bases:
    if (b / "last_epoch_predictions_train.json").exists():
        base = b
        break
else:
    base = Path.cwd()
    print(f"⚠️ Couldn’t find JSONs in known locations. Using {base}. Update `base` if needed.")

paths = {
    "train":   base / "last_epoch_predictions_train.json",
    "valid":   base / "last_epoch_predictions_valid.json",
    "holdout": base / "last_epoch_predictions_holdout.json",
}

def load_pred_file(p: Path, split: str):
    """Load one predictions JSON and return (df, class_names or None)."""
    if not p.exists():
        return pd.DataFrame(), None

    d = json.loads(p.read_text())
    cls = d.get("class_names", [])
    df  = pd.DataFrame(d.get("samples", []))
    if df.empty:
        return df, cls

    df["split"] = split

    # Ensure helpful columns exist / are consistent
    if "true_label" not in df.columns and "true_idx" in df.columns and cls:
        df["true_label"] = df["true_idx"].apply(
            lambda i: cls[int(i)] if isinstance(i, (int, np.integer)) and 0 <= int(i) < len(cls) else str(i)
        )
    if "pred_label" not in df.columns and "pred_idx" in df.columns and cls:
        df["pred_label"] = df["pred_idx"].apply(
            lambda i: cls[int(i)] if isinstance(i, (int, np.integer)) and 0 <= int(i) < len(cls) else str(i)
        )

    df["correct"] = df["true_idx"] == df["pred_idx"]

    # Confidence helpers (guard against missing/empty probs)
    def _safe_pick(row, key_idx):
        try:
            probs = row["probs"]
            idx = int(row[key_idx])
            return float(probs[idx]) if isinstance(probs, (list, tuple)) and 0 <= idx < len(probs) else np.nan
        except Exception:
            return np.nan

    df["pred_conf"] = df.apply(lambda r: _safe_pick(r, "pred_idx"), axis=1)
    df["true_conf"] = df.apply(lambda r: _safe_pick(r, "true_idx"), axis=1)
    df["max_prob"]  = df["probs"].apply(lambda v: float(np.max(v)) if isinstance(v, (list, tuple)) and len(v) else np.nan)

    return df, cls

# Load any/all of the splits
dfs, classes = [], None
for split, p in paths.items():
    df, cls = load_pred_file(p, split)
    if not df.empty:
        dfs.append(df)
        if classes is None and cls:
            classes = cls

all_pred = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# Quick sanity print
if all_pred.empty:
    print("⚠️ No prediction JSONs found/loaded.")
else:
    print(
        all_pred.groupby(["split", "correct"])
        .size()
        .rename("n")
        .reset_index()
        .to_string(index=False)
    )

  split  correct      n
holdout    False  18934
holdout     True  67046
  train    False  13375
  train     True 205390
  valid    False   9110
  valid     True  84690


In [19]:
# Extract site code like b07, c06, k04, e02
all_pred["site"] = (
    all_pred["filename"]
    .str.extract(r"_([a-z]\d{2})_", expand=False)
)
all_pred

,filename,true_idx,pred_idx,true_label,pred_label,probs,split,correct,pred_conf,true_conf,max_prob,site
0,08010440_b09_baboon.jpg,1,1,baboon,baboon,"[8.561577851651236e-05, 0.9972834587097168, 8....",train,True,0.997283,0.997283,0.997283,b09
1,2016 09 28 09 09 41_c06_oribi.jpg,5,5,oribi,oribi,"[0.0002549849741626531, 0.0009470327640883625,...",train,True,0.981533,0.981533,0.981533,c06
2,09080133_k04_baboon.jpg,1,1,baboon,baboon,"[0.02487700618803501, 0.9707638025283813, 3.89...",train,True,0.970764,0.970764,0.970764,k04
3,02160682_b07_waterbuck.jpg,0,0,waterbuck,waterbuck,"[0.9996296167373657, 1.3204116839915514e-05, 0...",train,True,0.999630,0.999630,0.999630,b07
4,2016 10 10 13 08 55_k06_warthog.jpg,2,2,warthog,warthog,"[2.8355521862977184e-05, 0.0002327105758013203...",train,True,0.999256,0.999256,0.999256,k06
...,...,...,...,...,...,...,...,...,...,...,...,...
398540,12050385_e02_impala.jpg,3,3,impala,impala,"[0.0071000768803060055, 0.0023746448568999767,...",holdout,True,0.989230,0.989230,0.989230,e02
398541,12200855_g08_nyala.jpg,8,8,nyala,nyala,"[1.220321973960381e-05, 3.775173900066875e-05,...",holdout,True,0.995053,0.995053,0.995053,g08
398542,EK002833_e06_warthog.jpg,2,2,warthog,warthog,"[2.7868883989867754e-05, 7.610805710100976e-07...",holdout,True,0.999905,0.999905,0.999905,e06
398543,02130732_d07_warthog.jpg,2,2,warthog,warthog,"[0.00033423927379772067, 4.694066956290044e-05...",holdout,True,0.999607,0.999607,0.999607,d07


## Grouped by Ground Label

In [17]:
import numpy as np
import pandas as pd
import altair as alt

THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]


def compute_review_table(all_pred: pd.DataFrame, split: str, thresholds=THRESHOLDS) -> pd.DataFrame:
    cols = [
        "true_label","threshold","split",
        "n_total",
        "n_correct_total","n_review","review_frac",
        "n_kept_correct","n_kept_wrong","acc_at_t"
    ]

    if all_pred is None or all_pred.empty:
        return pd.DataFrame(columns=cols)

    df = all_pred.copy()
    df["split"] = df["split"].astype(str).str.lower()
    df = df[df["split"] == split.lower()].copy()
    if df.empty:
        return pd.DataFrame(columns=cols)

    df["true_label"] = df["true_label"].astype(str)
    df["pred_label"] = df["pred_label"].astype(str)
    df["correct"] = df["correct"].astype(bool)

    df["max_prob"] = df["probs"].apply(
        lambda v: float(np.max(v)) if isinstance(v, (list, tuple, np.ndarray)) and len(v) else np.nan
    )

    df = df[df["max_prob"].notna()].copy()
    if df.empty:
        return pd.DataFrame(columns=cols)

    denom_total = df.groupby("true_label").size().rename("n_total")
    correct_df = df[df["correct"]].copy()
    denom_correct = correct_df.groupby("true_label").size().rename("n_correct_total")

    rows = []
    for t in thresholds:
        t = float(t)

        n_review = (
            correct_df[correct_df["max_prob"] < t]
            .groupby("true_label")
            .size()
            .rename("n_review")
        )

        kept = df[df["max_prob"] >= t]
        n_kept_correct = (
            kept[kept["correct"]]
            .groupby("true_label")
            .size()
            .rename("n_kept_correct")
        )
        n_kept_wrong = (
            kept[~kept["correct"]]
            .groupby("true_label")
            .size()
            .rename("n_kept_wrong")
        )

        out = pd.concat(
            [denom_total, denom_correct, n_review, n_kept_correct, n_kept_wrong],
            axis=1
        ).fillna(0)

        for c in ["n_total","n_correct_total","n_review","n_kept_correct","n_kept_wrong"]:
            out[c] = out[c].astype(int)

        out["review_frac"] = np.where(
            out["n_correct_total"] > 0,
            out["n_review"] / out["n_correct_total"],
            np.nan
        )

        denom_acc = out["n_kept_correct"] + out["n_kept_wrong"]
        out["acc_at_t"] = np.where(
            denom_acc > 0,
            out["n_kept_correct"] / denom_acc,
            np.nan
        )

        out["threshold"] = t
        out["split"] = split
        rows.append(out.reset_index())

    return pd.concat(rows, ignore_index=True).sort_values(["true_label", "threshold"])


def build_faceted_chart_same_y(
    tbl: pd.DataFrame,
    split: str,
    columns: int = 4,
    width: int = 180,
    height: int = 120
):
    if tbl is None or tbl.empty:
        return alt.Chart(pd.DataFrame({"msg":[f"No data for split={split}"]})).mark_text().encode(text="msg:N")

    class_order = (
        tbl.drop_duplicates("true_label")
           .sort_values("n_total", ascending=False)["true_label"]
           .tolist()
    )

    base = alt.Chart(tbl).properties(width=width, height=height)

    review_line = base.mark_line(point=True).encode(
        x=alt.X("threshold:Q", title="Threshold"),
        y=alt.Y("review_frac:Q", title="Rate", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("true_label:N", legend=None, scale=alt.Scale(scheme="tableau20"))
    )

    acc_line = base.mark_line(point=True, strokeDash=[2, 2]).encode(
        x="threshold:Q",
        y=alt.Y("acc_at_t:Q", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("true_label:N", legend=None, scale=alt.Scale(scheme="tableau20"))
    )

    n_text = (
        base.transform_aggregate(n_total="max(n_total)", groupby=["true_label"])
        .transform_calculate(label="'N=' + format(datum.n_total, ',')")
        .mark_text(align="center", baseline="middle", fontSize=12, opacity=0.85)
        .encode(
            text="label:N",
            x=alt.value(width * 0.55),
            y=alt.value(height * 0.55)
        )
    )

    faceted = (
        alt.layer(review_line, acc_line, n_text)
        .facet(
            facet=alt.Facet("true_label:N", sort=class_order, title=None),
            columns=columns
        )
        .properties(title=f"{split} — Solid: review fraction | Dotted: accuracy")
    )

    # ── Legend ──────────────────────────────────────────────
    legend_data = pd.DataFrame({
        "metric": ["Review fraction", "Accuracy"],
        "style": ["solid", "dotted"],
        "y": [2, 1]
    })

    legend_base = alt.Chart(legend_data).properties(width=220, height=120)

    legend_lines = legend_base.mark_line(strokeWidth=3).encode(
        x=alt.value(0),
        x2=alt.value(40),
        y=alt.Y("y:Q", axis=None),
        strokeDash=alt.StrokeDash(
            "style:N",
            scale=alt.Scale(domain=["solid", "dotted"], range=[[1, 0], [4, 2]]),
            legend=None
        )
    )

    legend_text = legend_base.mark_text(
        align="left", baseline="middle", dx=50, fontSize=13
    ).encode(
        y=alt.Y("y:Q", axis=None),
        text="metric:N"
    )

    legend_title = alt.Chart(
        pd.DataFrame({"t": ["Legend"]})
    ).mark_text(
        align="left", fontSize=14, fontWeight="bold"
    ).encode(text="t:N").properties(height=30)

    legend_panel = alt.vconcat(
        legend_title,
        alt.layer(legend_lines, legend_text),
        spacing=10
    )

    return alt.hconcat(faceted, legend_panel, spacing=20)


# ── Example usage ──────────────────────────────────────────
splits = ["train", "valid", "holdout"]
charts = {}

for sp in splits:
    tbl = compute_review_table(all_pred, sp)
    ch = build_faceted_chart_same_y(tbl, sp)
    charts[sp] = ch

    out_html = f"/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/review_and_accuracy_same_y_{sp}.html"
    ch.save(out_html)
    print("Saved:", out_html)

Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/review_and_accuracy_same_y_train.html
Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/review_and_accuracy_same_y_valid.html
Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/review_and_accuracy_same_y_holdout.html


In [20]:
import altair as alt
import numpy as np
import pandas as pd

def _filter_split(df: pd.DataFrame, split: str) -> pd.DataFrame:
    if split.lower() == "all":
        return df.copy()
    out = df[df["split"].str.lower() == split.lower()].copy()
    return out

def panel_counts(all_pred: pd.DataFrame, split: str = "all") -> pd.DataFrame:
    """Counts per (true_label, Correctness) for the chosen split."""
    df = _filter_split(all_pred, split)
    if df.empty:
        return pd.DataFrame(columns=["true_label", "Correctness", "n"])
    df = df.copy()
    df["Correctness"] = np.where(df["correct"], "Correct", "Incorrect")
    return (
        df.groupby(["true_label", "Correctness"])
          .size()
          .rename("n")
          .reset_index()
          .sort_values(["true_label", "Correctness"])
    )

def pair_counts(all_pred: pd.DataFrame, split: str = "all") -> pd.DataFrame:
    """Counts per (true_label, pred_label) for the chosen split."""
    df = _filter_split(all_pred, split)
    if df.empty:
        return pd.DataFrame(columns=["true_label", "pred_label", "n"])
    return (
        df.groupby(["true_label", "pred_label"])
          .size()
          .rename("n")
          .reset_index()
          .sort_values(["true_label", "n"], ascending=[True, False])
    )

def build_maxprob_chart(
    all_pred: pd.DataFrame,
    split: str = "all",
    binsize: float = 0.05,
    threshold_val: float = 0.90,
    width: int = 220,
    height: int = 120,
):
    """
    Faceted histograms of max_prob per class.
    Left column = Correct, Right column = Incorrect.
    Aligned by true_label across columns. Each panel shows N for that subset.

    Adds:
      - Threshold slider (red dashed rule)
      - Site dropdown filter (All + each site)
    """
    # Safety
    if all_pred is None or all_pred.empty:
        raise ValueError("all_pred is empty. Load your JSON(s) first.")

    alt.data_transformers.disable_max_rows()
    try:
        alt.renderers.enable("mimetype")
    except Exception:
        alt.renderers.enable("html")

    # Filter split
    df = _filter_split(all_pred, split)
    if df.empty:
        raise ValueError(f"No rows for split='{split}'.")
    df = df.copy()

    # Ensure types & helper col
    df["true_label"] = df["true_label"].astype(str)
    df["pred_label"] = df["pred_label"].astype(str)
    df["correct"]    = df["correct"].astype(bool)
    df["max_prob"]   = df["max_prob"].astype(float)
    df["Correctness"] = np.where(df["correct"], "Correct", "Incorrect")

    # Make sure site exists (if you already created it earlier, this is harmless)
    if "site" not in df.columns:
        df["site"] = df["filename"].astype(str).str.extract(r"_([a-z]\d{2})_", expand=False)

    # Row order by frequency
    class_order = df["true_label"].value_counts().index.tolist()

    # Threshold slider param
    threshold = alt.param(
        name="threshold", value=float(threshold_val),
        bind=alt.binding_range(min=0.0, max=1.0, step=0.01, name="Threshold")
    )

    # Site dropdown param (All + sorted sites)
    sites = sorted([s for s in df["site"].dropna().unique().tolist()])
    site_param = alt.param(
        name="site",
        value="All",
        bind=alt.binding_select(options=["All"] + sites, name="Site ")
    )

    # Base bars (symlog y to avoid log(0) issues)
    bars = (
        alt.Chart()
          .mark_bar()
          .encode(
              x=alt.X(
                  "max_prob:Q",
                  bin=alt.Bin(step=binsize),
                  scale=alt.Scale(domain=[0, 1]),
                  title="max_prob"
              ),
              y=alt.Y(
                  "count():Q",
                  scale=alt.Scale(type="symlog"),
                  title="count (symlog)"
              ),
          )
          .properties(width=width, height=height)
    )

    # Movable dashed rule (threshold)
    rule = (
        alt.Chart()
          .transform_calculate(xpos="threshold")
          .mark_rule(color="red", strokeDash=[5, 5])
          .encode(
              x=alt.X("xpos:Q", scale=alt.Scale(domain=[0, 1]), title="max_prob")
          )
          .properties(width=width, height=height)
    )

    # Panel count overlays (N=…) — one for Correct, one for Incorrect
    text_correct = (
        alt.Chart()
          .transform_filter(alt.datum.Correctness == "Correct")
          .transform_aggregate(n="count()", groupby=["true_label"])
          .mark_text(align="left", baseline="top", dx=4, dy=4)
          .encode(text=alt.Text("n:Q"))
          .properties(width=width, height=height)
    )
    text_incorrect = (
        alt.Chart()
          .transform_filter(alt.datum.Correctness == "Incorrect")
          .transform_aggregate(n="count()", groupby=["true_label"])
          .mark_text(align="left", baseline="top", dx=4, dy=4)
          .encode(text=alt.Text("n:Q"))
          .properties(width=width, height=height)
    )

    # Shared site filter expression: include all when "All" is selected
    site_filter = (site_param == "All") | (alt.datum.site == site_param)

    # Left = Correct
    left = (
        alt.layer(
            bars.transform_filter(alt.datum.Correctness == "Correct"),
            rule,
            text_correct,
            data=df,
        )
        .transform_filter(site_filter)
        .facet(row=alt.Row("true_label:N", sort=class_order, title=None))
        .properties(title="Correct")
    )

    # Right = Incorrect
    right = (
        alt.layer(
            bars.transform_filter(alt.datum.Correctness == "Incorrect"),
            rule,
            text_incorrect,
            data=df,
        )
        .transform_filter(site_filter)
        .facet(row=alt.Row("true_label:N", sort=class_order, title=None))
        .properties(title="Incorrect")
    )

    chart = (
        alt.hconcat(left, right)
          .resolve_scale(y="independent")
          .add_params(threshold, site_param)
          .properties(title=f"max_prob histograms — split={split}")
    )
    return chart


# ── EXAMPLE ─────────────────────────────────────────────────────────────
# Make sure you already created all_pred["site"] once, OR let the function extract it.
chart = build_maxprob_chart(all_pred, split="holdout", binsize=0.05, threshold_val=0.90)

out_html = "/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/maxprob_histograms_holdout.html"
chart.save(out_html)

pc = panel_counts(all_pred, split="holdout")
pairs = pair_counts(all_pred, split="holdout")
print(pc.head(), "\n", pairs.head())

  true_label Correctness      n
0     baboon     Correct  12178
1     baboon   Incorrect   1806
2    buffalo     Correct     97
3    buffalo   Incorrect    278
4   bushbuck     Correct   5619 
    true_label pred_label      n
0      baboon     baboon  12178
15     baboon    warthog   1031
16     baboon  waterbuck    287
7      baboon     impala    231
2      baboon   bushbuck     68


## Predicted Label Based

In [43]:
def build_revision_fraction_chart_predlabel(
    tbl: pd.DataFrame,
    split: str,
    columns: int = 4,
    width: int = 180,
    height: int = 120
):
    """
    Faceted by pred_label.
    Solid line = review_frac (fraction needing review among correct predictions
                              for that predicted label).
    N = total predictions as that label (n_total).
    """
    if tbl is None or tbl.empty:
        return alt.Chart(pd.DataFrame({"msg":[f"No data for split={split}"]})) \
                 .mark_text().encode(text="msg:N")

    # Order facets by how often the model predicts the label
    order = (
        tbl.drop_duplicates("pred_label")
           .sort_values("n_total", ascending=False)["pred_label"]
           .tolist()
    )

    base = alt.Chart(tbl).properties(width=width, height=height)

    # ── Solid line: fraction needing review ─────────────────
    review_line = base.mark_line(point=True).encode(
        x=alt.X("threshold:Q", title="Threshold"),
        y=alt.Y(
            "review_frac:Q",
            title="Fraction needing review",
            axis=alt.Axis(format="%"),
            scale=alt.Scale(domain=[0, 1])
        ),
        color=alt.Color(
            "pred_label:N",
            legend=None,
            scale=alt.Scale(scheme="tableau20")
        ),
        tooltip=[
            "pred_label:N",
            "threshold:Q",
            "n_total:Q",
            "n_correct_total:Q",
            "n_review:Q",
            alt.Tooltip("review_frac:Q", format=".2%")
        ],
    )

    # ── Center N label ──────────────────────────────────────
    n_text = (
        base.transform_aggregate(n_total="max(n_total)", groupby=["pred_label"])
            .transform_calculate(label="'N=' + format(datum.n_total, ',')")
            .mark_text(
                align="center",
                baseline="middle",
                fontSize=12,
                opacity=0.85
            )
            .encode(
                text="label:N",
                x=alt.value(width * 0.55),
                y=alt.value(height * 0.55)
            )
    )

    chart = (
        alt.layer(review_line, n_text)
        .facet(
            facet=alt.Facet("pred_label:N", sort=order, title=None),
            columns=columns
        )
        .properties(
            title=f"{split} — predicted-label view (fraction needing review)"
        )
    )
    return chart

In [44]:
tbl_train = compute_review_table(all_pred, "holdout", thresholds=THRESHOLDS)  # your pred-label version
ch_train = build_revision_fraction_chart_predlabel(tbl_train, "train", columns=4)
ch_train.save("/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/predlabel_revision_fraction_holdout.html")

In [45]:
import altair as alt
import numpy as np
import pandas as pd

def _filter_split(df: pd.DataFrame, split: str) -> pd.DataFrame:
    if split.lower() == "all":
        return df.copy()
    out = df[df["split"].str.lower() == split.lower()].copy()
    return out

def build_maxprob_chart_predlabel(
    all_pred: pd.DataFrame,
    split: str = "all",
    binsize: float = 0.05,
    threshold_val: float = 0.90,
    width: int = 220,
    height: int = 120,
):
    """
    Faceted histograms of max_prob per *predicted* label.
    Left column = Correct, Right column = Incorrect.
    Aligned by pred_label across columns. Each panel shows N for that subset.

    Adds:
      - Threshold slider (red dashed rule)
      - Site dropdown filter (All + each site)
      - Symlog y-scale
      - Same styling/structure as the true-label version
    """
    if all_pred is None or all_pred.empty:
        raise ValueError("all_pred is empty. Load your JSON(s) first.")

    alt.data_transformers.disable_max_rows()
    try:
        alt.renderers.enable("mimetype")
    except Exception:
        alt.renderers.enable("html")

    # Filter split
    df = _filter_split(all_pred, split)
    if df.empty:
        raise ValueError(f"No rows for split='{split}'.")
    df = df.copy()

    # Ensure types & helper col
    df["true_label"] = df["true_label"].astype(str)
    df["pred_label"] = df["pred_label"].astype(str)
    df["correct"]    = df["correct"].astype(bool)

    # If max_prob isn't present / correct type, compute or cast it
    if "max_prob" not in df.columns:
        df["max_prob"] = df["probs"].apply(
            lambda v: float(np.max(v)) if isinstance(v, (list, tuple, np.ndarray)) and len(v) else np.nan
        )
    df["max_prob"] = df["max_prob"].astype(float)

    df["Correctness"] = np.where(df["correct"], "Correct", "Incorrect")

    # Make sure site exists
    if "site" not in df.columns:
        df["site"] = df["filename"].astype(str).str.extract(r"_([a-z]\d{2})_", expand=False)

    # Row order by frequency of PREDICTED label
    class_order = df["pred_label"].value_counts().index.tolist()

    # Threshold slider param
    threshold = alt.param(
        name="threshold", value=float(threshold_val),
        bind=alt.binding_range(min=0.0, max=1.0, step=0.01, name="Threshold")
    )

    # Site dropdown param
    sites = sorted([s for s in df["site"].dropna().unique().tolist()])
    site_param = alt.param(
        name="site",
        value="All",
        bind=alt.binding_select(options=["All"] + sites, name="Site ")
    )

    # Base bars (symlog y)
    bars = (
        alt.Chart()
          .mark_bar()
          .encode(
              x=alt.X(
                  "max_prob:Q",
                  bin=alt.Bin(step=binsize),
                  scale=alt.Scale(domain=[0, 1]),
                  title="max_prob"
              ),
              y=alt.Y(
                  "count():Q",
                  scale=alt.Scale(type="symlog"),
                  title="count (symlog)"
              ),
          )
          .properties(width=width, height=height)
    )

    # Movable threshold rule
    rule = (
        alt.Chart()
          .transform_calculate(xpos="threshold")
          .mark_rule(color="red", strokeDash=[5, 5])
          .encode(
              x=alt.X("xpos:Q", scale=alt.Scale(domain=[0, 1]), title="max_prob")
          )
          .properties(width=width, height=height)
    )

    # Panel count overlays (N=…) for Correct/Incorrect, grouped by pred_label
    text_correct = (
        alt.Chart()
          .transform_filter(alt.datum.Correctness == "Correct")
          .transform_aggregate(n="count()", groupby=["pred_label"])
          .mark_text(align="left", baseline="top", dx=4, dy=4)
          .encode(text=alt.Text("n:Q"))
          .properties(width=width, height=height)
    )
    text_incorrect = (
        alt.Chart()
          .transform_filter(alt.datum.Correctness == "Incorrect")
          .transform_aggregate(n="count()", groupby=["pred_label"])
          .mark_text(align="left", baseline="top", dx=4, dy=4)
          .encode(text=alt.Text("n:Q"))
          .properties(width=width, height=height)
    )

    # Shared site filter (All => include everything)
    site_filter = (site_param == "All") | (alt.datum.site == site_param)

    # Left = Correct (pred-label row facets)
    left = (
        alt.layer(
            bars.transform_filter(alt.datum.Correctness == "Correct"),
            rule,
            text_correct,
            data=df,
        )
        .transform_filter(site_filter)
        .facet(row=alt.Row("pred_label:N", sort=class_order, title=None))
        .properties(title="Correct")
    )

    # Right = Incorrect (pred-label row facets)
    right = (
        alt.layer(
            bars.transform_filter(alt.datum.Correctness == "Incorrect"),
            rule,
            text_incorrect,
            data=df,
        )
        .transform_filter(site_filter)
        .facet(row=alt.Row("pred_label:N", sort=class_order, title=None))
        .properties(title="Incorrect")
    )

    chart = (
        alt.hconcat(left, right)
          .resolve_scale(y="independent")
          .add_params(threshold, site_param)
          .properties(title=f"max_prob histograms by predicted label — split={split}")
    )
    return chart

In [48]:
ch = build_maxprob_chart_predlabel(all_pred, split="holdout", binsize=0.05, width=240, height=120)
out_html = "/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/predlabel_maxprob_holdout.html"
ch.save(out_html)

## Metrics

In [58]:
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
try:
    alt.renderers.enable("mimetype")
except Exception:
    alt.renderers.enable("html")

THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]


def compute_per_class_threshold_metrics(all_pred: pd.DataFrame, split: str, thresholds=THRESHOLDS) -> pd.DataFrame:
    """
    Per-class (one-vs-rest) metrics on the KEPT subset, per threshold.
    Keep rule: max_prob >= threshold

    For each class c and threshold t, computed on kept rows only:
      TP = pred=c & true=c
      FP = pred=c & true!=c
      FN = pred!=c & true=c
      TN = pred!=c & true!=c

      precision = TP/(TP+FP)
      recall    = TP/(TP+FN)
      f1        = 2PR/(P+R)
      accuracy  = (TP+TN)/(TP+FP+FN+TN)   # binary accuracy one-vs-rest on kept set
    """
    df = all_pred.copy()
    df["split"] = df["split"].astype(str).str.lower()
    df = df[df["split"] == split.lower()].copy()
    if df.empty:
        return pd.DataFrame(columns=["split","threshold","label","metric","value","tp","fp","fn","tn","n_kept"])

    df["true_label"] = df["true_label"].astype(str)
    df["pred_label"] = df["pred_label"].astype(str)

    # ensure max_prob exists
    if "max_prob" not in df.columns:
        df["max_prob"] = df["probs"].apply(
            lambda v: float(np.max(v)) if isinstance(v, (list, tuple, np.ndarray)) and len(v) else np.nan
        )
    df["max_prob"] = df["max_prob"].astype(float)
    df = df[df["max_prob"].notna()].copy()
    if df.empty:
        return pd.DataFrame(columns=["split","threshold","label","metric","value","tp","fp","fn","tn","n_kept"])

    # classes to evaluate (use union so we include classes that appear in pred or true)
    labels = sorted(set(df["true_label"].unique()).union(set(df["pred_label"].unique())))

    rows = []
    for t in thresholds:
        t = float(t)
        kept = df[df["max_prob"] >= t].copy()
        n_kept = len(kept)
        if n_kept == 0:
            # no kept data → all NaN
            for lab in labels:
                for metric in ["accuracy","precision","recall","f1"]:
                    rows.append({
                        "split": split, "threshold": t, "label": lab,
                        "metric": metric, "value": np.nan,
                        "tp": 0, "fp": 0, "fn": 0, "tn": 0, "n_kept": 0
                    })
            continue

        true = kept["true_label"].values
        pred = kept["pred_label"].values

        for lab in labels:
            pred_is = (pred == lab)
            true_is = (true == lab)

            tp = int(np.sum(pred_is & true_is))
            fp = int(np.sum(pred_is & (~true_is)))
            fn = int(np.sum((~pred_is) & true_is))
            tn = int(np.sum((~pred_is) & (~true_is)))

            precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            recall    = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            f1 = (2 * precision * recall / (precision + recall)) if (
                precision is not np.nan and recall is not np.nan and (precision + recall) > 0
            ) else np.nan
            accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else np.nan

            rows.extend([
                {"split": split, "threshold": t, "label": lab, "metric": "accuracy",  "value": accuracy,  "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n_kept": n_kept},
                {"split": split, "threshold": t, "label": lab, "metric": "precision", "value": precision, "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n_kept": n_kept},
                {"split": split, "threshold": t, "label": lab, "metric": "recall",    "value": recall,    "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n_kept": n_kept},
                {"split": split, "threshold": t, "label": lab, "metric": "f1",        "value": f1,        "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n_kept": n_kept},
            ])

    return pd.DataFrame(rows)


def plot_per_class_metrics(per_class_long: pd.DataFrame, split: str, columns: int = 4, width: int = 180, height: int = 120):
    """
    4 lines per class facet (accuracy, precision, recall, f1) vs threshold.
    """
    if per_class_long is None or per_class_long.empty:
        return alt.Chart(pd.DataFrame({"msg":[f"No data for split={split}"]})).mark_text().encode(text="msg:N")

    # order facets by how often the class appears as true label (optional)
    # you can change this to pred_label frequency if you want
    order = (
        per_class_long[per_class_long["metric"] == "accuracy"]
        .groupby("label")["tp"].max()
        .sort_values(ascending=False)
        .index.tolist()
    )

    chart = (
        alt.Chart(per_class_long)
        .mark_line(point=True)
        .encode(
            x=alt.X("threshold:Q", title="Threshold"),
            y=alt.Y("value:Q", title="Metric (kept only)", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0,1])),
            color=alt.Color("metric:N", title="Metric"),
            tooltip=[
                "label:N", "metric:N", "threshold:Q",
                alt.Tooltip("value:Q", format=".2%"),
                alt.Tooltip("tp:Q"), alt.Tooltip("fp:Q"), alt.Tooltip("fn:Q"), alt.Tooltip("tn:Q"),
                alt.Tooltip("n_kept:Q", title="kept total"),
            ],
        )
        .properties(width=width, height=height)
        .facet(
            facet=alt.Facet("label:N", sort=order, title=None),
            columns=columns
        )
        .properties(title=f"{split}: per-class metrics vs threshold (kept only)")
    )

    return chart


# ---- Make charts + save ----
splits = ["train", "valid", "holdout"]

for sp in splits:
    per_class = compute_per_class_threshold_metrics(all_pred, sp, thresholds=THRESHOLDS)

    ch = plot_per_class_metrics(per_class, sp, columns=4, width=180, height=120)

    out_html = f"/Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/per_class_metrics_vs_threshold_{sp}.html"
    ch.save(out_html)
    print("Saved:", out_html)

# show one inline if you want
# ch

Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/per_class_metrics_vs_threshold_train.html
Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/per_class_metrics_vs_threshold_valid.html
Saved: /Users/sarahabdelazim/Desktop/Kaitlyn/kaitlyn_catalyst/notebooks/per_class_metrics_vs_threshold_holdout.html
